# 队列（Queue）完全教程：从零到精通

**适用对象**：数据结构初学者 | **语言**：C++ | **预计学时**：~1.5h

---

## 第一章：队列的基本概念

### 1.1 什么是队列？

想象你在食堂排队打饭：**先来的人先打饭，后来的人排到队尾**。这就是队列的核心思想——**先进先出（FIFO, First In First Out）**。

队列是一种**操作受限的线性表**，它只允许：
- 在**队尾（rear）**插入元素 —— 称为**入队（enqueue）**
- 在**队头（front）**删除元素 —— 称为**出队（dequeue）**

```
            ┌───┬───┬───┬───┬───┐
  队头 front│ A │ B │ C │ D │ E │ 队尾 rear
            └───┴───┴───┴───┴───┘

  出队 ← 【A】 B  C  D  E  ← 入队
          front            rear
```

### 1.2 队列 vs 栈——一对"兄弟"

| 特性     | 栈（Stack）   | 队列（Queue） |
| -------- | ------------- | ------------- |
| 原则     | 后进先出 LIFO | 先进先出 FIFO |
| 插入     | 栈顶 push     | 队尾 enqueue  |
| 删除     | 栈顶 pop      | 队头 dequeue  |
| 日常类比 | 一摞盘子      | 排队买饭      |

### 1.3 队列的抽象数据类型（ADT）

一个队列至少需要支持以下操作：

```
ADT Queue {
    InitQueue()    —— 初始化一个空队列
    IsEmpty()      —— 判断队列是否为空
    IsFull()       —— 判断队列是否已满（顺序存储时）
    EnQueue(x)     —— 将元素 x 入队（插到队尾）
    DeQueue()      —— 队头元素出队
    GetFront()     —— 获取队头元素（不删除）
}
```

---

## 第二章：队列的顺序实现（数组实现）

### 2.1 最朴素的想法

用一个数组 `data[]` 来存储队列元素，用两个变量 `front` 和 `rear` 分别指示队头和队尾。

In [ ]:
#define MAXSIZE 5

In [ ]:
struct Queue {
    int data[MAXSIZE];
    int front;  // 指向队头元素
    int rear;   // 指向队尾元素的下一个位置
};

> **约定**：`front` 指向队头元素，`rear` 指向队尾元素的**下一个位置**（即下一个要插入的位置）。这是王道教材中最常用的约定。

#### 初始状态

```
front = 0, rear = 0（队空）

索引：  0   1   2   3   4
      ┌───┬───┬───┬───┬───┐
      │   │   │   │   │   │
      └───┴───┴───┴───┴───┘
       ↑
     front
     rear
```

#### 入队过程演示

```
EnQueue(10): data[rear] = 10; rear++;

索引：  0   1   2   3   4
      ┌────┬───┬───┬───┬───┐
      │ 10 │   │   │   │   │
      └────┴───┴───┴───┴───┘
       ↑    ↑
     front rear

EnQueue(20): data[rear] = 20; rear++;

索引：  0   1   2   3   4
      ┌────┬────┬───┬───┬───┐
      │ 10 │ 20 │   │   │   │
      └────┴────┴───┴───┴───┘
       ↑         ↑
     front      rear

EnQueue(30), EnQueue(40), EnQueue(50):

索引：  0   1   2   3   4
      ┌────┬────┬────┬────┬────┐
      │ 10 │ 20 │ 30 │ 40 │ 50 │
      └────┴────┴────┴────┴────┘
       ↑                         ↑
     front                     rear = 5（已满）
```

#### 出队过程演示

```
DeQueue(): 取出 data[front]=10, front++;

索引：  0   1   2   3   4
      ┌────┬────┬────┬────┬────┐
      │    │ 20 │ 30 │ 40 │ 50 │
      └────┴────┴────┴────┴────┘
            ↑                    ↑
          front                rear = 5

DeQueue(): 取出 data[front]=20, front++;

索引：  0   1   2   3   4
      ┌────┬────┬────┬────┬────┐
      │    │    │ 30 │ 40 │ 50 │
      └────┴────┴────┴────┴────┘
                 ↑               ↑
               front           rear = 5
```

### 2.2 假溢出问题 ⚠️

现在问题来了！上面的状态中，`rear = 5 = MAXSIZE`，看起来队列满了。但是索引 0 和 1 的位置明明是**空的**！

```
索引：  0   1   2   3   4
      ┌────┬────┬────┬────┬────┐
      │空！│空！│ 30 │ 40 │ 50 │
      └────┴────┴────┴────┴────┘
                 ↑               ↑
               front           rear = 5 → "满了"？？？
```

这就是**假溢出（False Overflow）**：数组前面明明有空间，但 `rear` 已经到达数组末尾，无法再入队。

**解决方案**：把数组的头和尾"接起来"，形成一个**逻辑上的环** → **循环队列**！

---

## 第三章：循环队列（Circular Queue）—— 重点中的重点 ⭐⭐⭐

### 3.1 核心思想

将数组想象成一个**圆环**，当 `rear` 到达数组末尾时，如果前面有空位，就"绕回"到数组开头。

实现方式：**取模运算**！

```
rear = (rear + 1) % MAXSIZE;
front = (front + 1) % MAXSIZE;
```

**图示理解**：

```
        把线性数组：
        索引：0  1  2  3  4

        想象成环形：

              [0]
           [4]   [1]
           [3]   [2]

        rear 到了 4 之后，下一步是 (4+1)%5 = 0，回到开头！
```

### 3.2 循环队列的判空和判满

这里有一个经典难题。看这两种状态：

```
状态1（队空）：front == rear

              [0]
           [4]   [1]        全空
           [3]   [2]
            ↑
          front
          rear

状态2（队满）：front == rear  ← 一样的条件！？

              [0] ← A
         E ← [4]   [1] ← B     全满
         D ← [3]   [2] ← C
               ↑
             front
             rear（绕了一圈回来）
```

**问题**：`front == rear` 既可能表示队空，也可能表示队满，无法区分！

#### 解决方案（三种，考试必考）

**方案一：牺牲一个存储单元（最常用！）** ⭐

> 约定：`(rear + 1) % MAXSIZE == front` 时为队满。即 rear 的下一个位置是 front 时就算满了，这意味着永远空出一个位置不用。

```
队空条件：front == rear
队满条件：(rear + 1) % MAXSIZE == front
队列长度：(rear - front + MAXSIZE) % MAXSIZE
```

图示：
```
           MAXSIZE = 5，最多存 4 个元素

  假设 front=0, 依次入队 A,B,C,D：

  入A: data[0]=A, rear=(0+1)%5=1
  入B: data[1]=B, rear=(1+1)%5=2
  入C: data[2]=C, rear=(2+1)%5=3
  入D: data[3]=D, rear=(3+1)%5=4

  此时 (rear+1)%5 = (4+1)%5 = 0 == front → 队满！
  data[4] 空着没用，但逻辑上认为满了。

  索引：  0   1   2   3   4
        ┌───┬───┬───┬───┬───┐
        │ A │ B │ C │ D │   │  ← 位置4牺牲
        └───┴───┴───┴───┴───┘
         ↑                ↑
       front            rear
```

**方案二：增设 size 变量**

```
队空条件：size == 0
队满条件：size == MAXSIZE
```

这样 front == rear 时，只需看 size 就能区分是空还是满。可以充分利用所有 MAXSIZE 个空间。

**方案三：增设 tag 标志位**

```
tag = 0：上次操作是出队 → front == rear 意味着队空
tag = 1：上次操作是入队 → front == rear 意味着队满
```

### 3.3 完整代码：循环队列（方案一：牺牲一个空间）

In [ ]:
#include <iostream>
using namespace std;

#define MAXSIZE 6  // 数组大小为6，最多存5个元素

In [ ]:
struct CircularQueue {
    int data[MAXSIZE];
    int front;  // 指向队头元素
    int rear;   // 指向队尾元素的下一个位置
};

In [ ]:
// 1. 初始化
void InitQueue(CircularQueue &Q) {
    Q.front = 0;
    Q.rear = 0;
}

// 2. 判空

In [ ]:
bool IsEmpty(CircularQueue Q) {
    return Q.front == Q.rear;
}

// 3. 判满

In [ ]:
bool IsFull(CircularQueue Q) {
    return (Q.rear + 1) % MAXSIZE == Q.front;
}

// 4. 求队列长度

In [ ]:
int QueueLength(CircularQueue Q) {
    return (Q.rear - Q.front + MAXSIZE) % MAXSIZE;
}

// 5. 入队

In [ ]:
bool EnQueue(CircularQueue &Q, int x) {
    if (IsFull(Q)) {
        cout << "队列已满，无法入队！" << endl;
        return false;
    }
    Q.data[Q.rear] = x;              // 将元素放入队尾
    Q.rear = (Q.rear + 1) % MAXSIZE; // rear 后移（取模实现循环）
    return true;
}

// 6. 出队

In [ ]:
bool DeQueue(CircularQueue &Q, int &x) {
    if (IsEmpty(Q)) {
        cout << "队列为空，无法出队！" << endl;
        return false;
    }
    x = Q.data[Q.front];               // 取出队头元素
    Q.front = (Q.front + 1) % MAXSIZE; // front 后移（取模实现循环）
    return true;
}

// 7. 获取队头元素（不删除）

In [ ]:
bool GetFront(CircularQueue Q, int &x) {
    if (IsEmpty(Q)) {
        cout << "队列为空！" << endl;
        return false;
    }
    x = Q.data[Q.front];
    return true;
}

// 8. 打印队列（调试用）

In [ ]:
void PrintQueue(CircularQueue Q) {
    cout << "队列内容（从队头到队尾）：";
    int i = Q.front;
    while (i != Q.rear) {
        cout << Q.data[i] << " ";
        i = (i + 1) % MAXSIZE;
    }
    cout << endl;
    cout << "front=" << Q.front << ", rear=" << Q.rear 
         << ", length=" << QueueLength(Q) << endl;
}

In [ ]:
int main() {
    CircularQueue Q;
    InitQueue(Q);

    cout << "=== 入队测试 ===" << endl;
    EnQueue(Q, 10);
    EnQueue(Q, 20);
    EnQueue(Q, 30);
    EnQueue(Q, 40);
    EnQueue(Q, 50);
    PrintQueue(Q);
    // 输出：10 20 30 40 50, length=5

    cout << "\n=== 尝试入队第6个元素（应该失败）===" << endl;
    EnQueue(Q, 60);  // 队满，失败

    cout << "\n=== 出队两个元素 ===" << endl;
    int val;
    DeQueue(Q, val);
    cout << "出队元素：" << val << endl;  // 10
    DeQueue(Q, val);
    cout << "出队元素：" << val << endl;  // 20
    PrintQueue(Q);

    cout << "\n=== 再入队两个元素（循环利用空间）===" << endl;
    EnQueue(Q, 60);
    EnQueue(Q, 70);
    PrintQueue(Q);
    // 此时 60 和 70 应该被放到了数组前面的位置

    return 0;
}

In [ ]:
main();

**运行结果**：

```
=== 入队测试 ===
队列内容（从队头到队尾）：10 20 30 40 50 
front=0, rear=5, length=5

=== 尝试入队第6个元素（应该失败）===
队列已满，无法入队！

=== 出队两个元素 ===
出队元素：10
出队元素：20
队列内容（从队头到队尾）：30 40 50 
front=2, rear=5, length=3

=== 再入队两个元素（循环利用空间）===
队列内容（从队头到队尾）：30 40 50 60 70 
front=2, rear=1, length=5
```

注意最后 `front=2, rear=1`，rear 已经"绕"到了数组前面，这就是循环队列的魅力！

### 3.4 循环队列求长度的公式推导

为什么 `(rear - front + MAXSIZE) % MAXSIZE` 能正确求出长度？

```
情况1：rear >= front（没有绕圈）
  
  索引：  0   1   2   3   4   5
        ┌───┬───┬───┬───┬───┬───┐
        │   │   │ A │ B │ C │   │
        └───┴───┴───┴───┴───┴───┘
                 ↑           ↑
               front=2     rear=5

  长度 = rear - front = 5 - 2 = 3 ✓

情况2：rear < front（绕了圈）

  索引：  0   1   2   3   4   5
        ┌───┬───┬───┬───┬───┬───┐
        │ C │ D │   │   │ A │ B │
        └───┴───┴───┴───┴───┴───┘
              ↑       ↑
            rear=2  front=4

  直接 rear - front = 2 - 4 = -2（负数！错误）
  加上 MAXSIZE：-2 + 6 = 4
  再取模：4 % 6 = 4 ✓（确实有 A,B,C,D 四个元素）

统一公式：(rear - front + MAXSIZE) % MAXSIZE
  当 rear >= front 时：(正数 + MAXSIZE) % MAXSIZE = 正数本身 ✓
  当 rear < front 时：(负数 + MAXSIZE) % MAXSIZE = 正确结果 ✓
```

---

## 第四章：队列的链式实现（链队列）

### 4.1 为什么需要链队列？

顺序队列（循环队列）有一个缺点：**大小固定**。如果事先不知道需要存多少元素，链式存储是更好的选择。

链队列 = 一个单链表 + 两个指针（front 指向队头，rear 指向队尾）。

### 4.2 链队列的结构

```
        front                              rear
          ↓                                  ↓
  ┌───┬───┐    ┌───┬───┐    ┌───┬───┐    ┌───┬───┐
  │   │ ──┼──→ │ A │ ──┼──→ │ B │ ──┼──→ │ C │ / │
  └───┴───┘    └───┴───┘    └───┴───┘    └───┴───┘
   头结点       第1个结点     第2个结点     第3个结点
```

> **带头结点 vs 不带头结点**：和单链表一样，带头结点可以统一空队和非空队的操作，代码更简洁。下面的代码以**带头结点**为例。

### 4.3 链队列的代码实现

In [ ]:
#include <iostream>
using namespace std;

// 链队列的结点

In [ ]:
struct LinkNode {
    int data;
    LinkNode *next;
};

In [ ]:
// 链队列（带头结点）
struct LinkQueue {
    LinkNode *front;  // 队头指针（指向头结点）
    LinkNode *rear;   // 队尾指针（指向最后一个实际结点）
};

In [ ]:
// 1. 初始化（带头结点）
void InitQueue(LinkQueue &Q) {
    // 创建头结点
    Q.front = new LinkNode;
    Q.front->next = nullptr;
    Q.rear = Q.front;  // 队空时，front 和 rear 都指向头结点
}

// 2. 判空

In [ ]:
bool IsEmpty(LinkQueue Q) {
    return Q.front == Q.rear;
    // 等价于：Q.front->next == nullptr
}

// 3. 入队（在队尾插入）

In [ ]:
void EnQueue(LinkQueue &Q, int x) {
    LinkNode *newNode = new LinkNode;
    newNode->data = x;
    newNode->next = nullptr;

    Q.rear->next = newNode;  // 新结点接到队尾后面
    Q.rear = newNode;        // rear 指向新的队尾
}

// 4. 出队（删除队头元素）

In [ ]:
bool DeQueue(LinkQueue &Q, int &x) {
    if (IsEmpty(Q)) {
        cout << "队列为空，无法出队！" << endl;
        return false;
    }

    LinkNode *p = Q.front->next;  // p 指向第一个实际结点（队头元素）
    x = p->data;
    Q.front->next = p->next;      // 头结点指向第二个结点

    // ⚠️ 关键：如果删除的是最后一个结点，rear 要指回头结点
    if (Q.rear == p) {
        Q.rear = Q.front;
    }

    delete p;
    return true;
}

// 5. 获取队头元素

In [ ]:
bool GetFront(LinkQueue Q, int &x) {
    if (IsEmpty(Q)) {
        cout << "队列为空！" << endl;
        return false;
    }
    x = Q.front->next->data;
    return true;
}

// 6. 打印队列

In [ ]:
void PrintQueue(LinkQueue Q) {
    cout << "队列内容：";
    LinkNode *p = Q.front->next;
    while (p != nullptr) {
        cout << p->data << " ";
        p = p->next;
    }
    cout << endl;
}

// 7. 销毁队列（释放所有内存）

In [ ]:
void DestroyQueue(LinkQueue &Q) {
    while (Q.front != nullptr) {
        LinkNode *p = Q.front;
        Q.front = Q.front->next;
        delete p;
    }
    Q.rear = nullptr;
}

In [ ]:
int main() {
    LinkQueue Q;
    InitQueue(Q);

    cout << "=== 入队测试 ===" << endl;
    EnQueue(Q, 100);
    EnQueue(Q, 200);
    EnQueue(Q, 300);
    EnQueue(Q, 400);
    PrintQueue(Q);
    // 输出：100 200 300 400

    cout << "\n=== 出队测试 ===" << endl;
    int val;
    DeQueue(Q, val);
    cout << "出队：" << val << endl;  // 100
    DeQueue(Q, val);
    cout << "出队：" << val << endl;  // 200
    PrintQueue(Q);
    // 输出：300 400

    cout << "\n=== 动态增长（链队列无大小限制）===" << endl;
    for (int i = 1; i <= 10; i++) {
        EnQueue(Q, i * 111);
    }
    PrintQueue(Q);

    DestroyQueue(Q);
    return 0;
}

In [ ]:
main();

### 4.4 不带头结点的链队列（对比）

不带头结点时，空队的处理需要特判：

In [ ]:
// 不带头结点的初始化

In [ ]:
void InitQueue(LinkQueue &Q) {
    Q.front = nullptr;
    Q.rear = nullptr;
}

// 不带头结点的入队

In [ ]:
void EnQueue(LinkQueue &Q, int x) {
    LinkNode *newNode = new LinkNode;
    newNode->data = x;
    newNode->next = nullptr;

    if (Q.front == nullptr) {
        // ⚠️ 特判：队列为空时，front 和 rear 都指向新结点
        Q.front = newNode;
        Q.rear = newNode;
    } else {
        Q.rear->next = newNode;
        Q.rear = newNode;
    }
}

// 不带头结点的出队

In [ ]:
bool DeQueue(LinkQueue &Q, int &x) {
    if (Q.front == nullptr) return false;

    LinkNode *p = Q.front;
    x = p->data;
    Q.front = p->next;

    if (Q.rear == p) {
        // ⚠️ 特判：删除最后一个元素后，rear 也要置空
        Q.front = nullptr;
        Q.rear = nullptr;
    }

    delete p;
    return true;
}

> **考试建议**：王道教材中带头结点和不带头结点都可能考，务必两种都会。关键区别就是**空队时的特殊处理**。

### 4.5 顺序队列 vs 链队列对比

| 特性      | 顺序队列（循环队列）   | 链队列               |
| --------- | ---------------------- | -------------------- |
| 存储空间  | 固定大小               | 动态分配             |
| 入队/出队 | O(1)                   | O(1)                 |
| 空间利用  | 牺牲一个位置（方案一） | 每个结点多一个指针域 |
| 适用场景  | 元素个数可预估         | 元素个数不确定       |
| 假溢出    | 用循环解决             | 不存在               |

---

## 第五章：双端队列（Double-Ended Queue / Deque）

### 5.1 什么是双端队列？

普通队列：只能队尾入、队头出。
双端队列：**两端都可以入队和出队**！

```
         ←入队      入队→
         ←出队      出队→
          ┌───┬───┬───┬───┬───┐
  前端 ←  │ A │ B │ C │ D │ E │  → 后端
          └───┴───┴───┴───┴───┘
```

### 5.2 双端队列的变体

#### （1）输入受限的双端队列

**只允许一端输入，两端都可以输出。**

```
              只能这端入队 →
         ←出队           出队→
          ┌───┬───┬───┬───┬───┐
  前端    │   │   │   │   │   │    后端
          └───┴───┴───┴───┴───┘
```

#### （2）输出受限的双端队列

**两端都可以输入，只允许一端输出。**

```
         ←入队           入队→
         ←出队（只能这端出队）
          ┌───┬───┬───┬───┬───┐
  前端    │   │   │   │   │   │    后端
          └───┴───┴───┴───┴───┘
```

### 5.3 考试经典题型：判断合法的输出序列

> **题目**：输入序列为 1, 2, 3, 4，判断哪些输出序列是合法的。

**对于普通队列**：只有 `1, 2, 3, 4` 这一种输出序列（先进先出，没得选）。

**对于输入受限的双端队列**：

```
输入只能从后端进入：1→, 2→, 3→, 4→
输出可以从两端出：←前端  后端→

示例：能否产生 4, 1, 3, 2？

操作序列：
  1. 入1（后端）：         [1]
  2. 入2（后端）：         [1, 2]
  3. 入3（后端）：         [1, 2, 3]
  4. 入4（后端）：         [1, 2, 3, 4]
  5. 出后端 → 4：          [1, 2, 3]         输出：4
  6. 出前端 → 1：          [2, 3]            输出：4, 1
  7. 出后端 → 3：          [2]               输出：4, 1, 3
  8. 出后端 → 2：          []                输出：4, 1, 3, 2  ✓
```

**对于输出受限的双端队列**：

```
输入可以从两端进入：←前端  后端→
输出只能从前端出：←前端

示例：能否产生 4, 2, 1, 3？

操作序列：
  1. 入1（前端）：          [1]
  2. 入2（前端）：          [2, 1]
  3. 入3（后端）：          [2, 1, 3]
  4. 入4（前端）：          [4, 2, 1, 3]
  5. 出前端 → 4：           [2, 1, 3]         输出：4
  6. 出前端 → 2：           [1, 3]            输出：4, 2
  7. 出前端 → 1：           [3]               输出：4, 2, 1
  8. 出前端 → 3：           []                输出：4, 2, 1, 3  ✓
```

### 5.4 判断技巧总结

| 队列类型         | 不可能产生的序列特征                              |
| ---------------- | ------------------------------------------------- |
| 普通队列         | 除 1,2,3,...,n 外的任何序列                       |
| 普通栈           | 存在 i<j<k，使得 a[j]<a[k]<a[i] 的序列            |
| 输入受限双端队列 | 能产生比栈更多的序列，但仍有限制                  |
| 输出受限双端队列 | 同上，与输入受限的限制不同                        |

> **考试秘诀**：如果题目问"哪个不是合法序列"，逐个模拟操作过程即可。不需要记公式，**手动模拟是最靠谱的方法**。

---

## 第六章：队列的应用

### 6.1 应用一：树的层序遍历（Level Order Traversal）

层序遍历是队列最经典的应用之一。思路：

```
1. 根结点入队
2. 循环：
   a. 队头元素出队，访问它
   b. 将它的左孩子入队（如果有）
   c. 将它的右孩子入队（如果有）
3. 直到队列为空
```

**手动演示**：

```
        1
       / \
      2   3
     / \   \
    4   5   6

步骤：
队列状态          出队元素    访问顺序
[1]               -           -
[2, 3]            1           1
[3, 4, 5]         2           1, 2
[4, 5, 6]         3           1, 2, 3
[5, 6]            4           1, 2, 3, 4
[6]               5           1, 2, 3, 4, 5
[]                6           1, 2, 3, 4, 5, 6

结果：1 2 3 4 5 6 （逐层从左到右）
```

**代码实现**：

In [ ]:
#include <iostream>
#include <queue>  // STL队列
using namespace std;

In [ ]:
struct TreeNode {
    int data;
    TreeNode *left, *right;
    TreeNode(int val) : data(val), left(nullptr), right(nullptr) {}
};

In [ ]:
void LevelOrder(TreeNode *root) {
    if (root == nullptr) return;

    queue<TreeNode*> Q;  // 创建队列
    Q.push(root);        // 根结点入队

    while (!Q.empty()) {
        TreeNode *node = Q.front();  // 取队头
        Q.pop();                     // 出队
        cout << node->data << " ";   // 访问

        if (node->left)  Q.push(node->left);   // 左孩子入队
        if (node->right) Q.push(node->right);  // 右孩子入队
    }
    cout << endl;
}

In [ ]:
int main() {
    // 构建上面的示例树
    TreeNode *root = new TreeNode(1);
    root->left = new TreeNode(2);
    root->right = new TreeNode(3);
    root->left->left = new TreeNode(4);
    root->left->right = new TreeNode(5);
    root->right->right = new TreeNode(6);

    cout << "层序遍历：";
    LevelOrder(root);
    // 输出：1 2 3 4 5 6

    return 0;
}

In [ ]:
main();

### 6.2 应用二：图的广度优先搜索（BFS）

BFS 本质上就是"图版本的层序遍历"。思路完全相同，只是多了一个**访问标记数组**来防止重复访问。

In [ ]:
#include <iostream>
#include <queue>
#include <vector>
using namespace std;

// 邻接表表示的图

In [ ]:
void BFS(vector<vector<int>> &adj, int start, int n) {
    vector<bool> visited(n, false);  // 访问标记
    queue<int> Q;

    visited[start] = true;
    Q.push(start);

    cout << "BFS 遍历顺序：";
    while (!Q.empty()) {
        int v = Q.front();
        Q.pop();
        cout << v << " ";

        // 将 v 的所有未访问邻居入队
        for (int neighbor : adj[v]) {
            if (!visited[neighbor]) {
                visited[neighbor] = true;
                Q.push(neighbor);
            }
        }
    }
    cout << endl;
}

In [ ]:
int main() {
    int n = 6;  // 6 个顶点 (0~5)
    vector<vector<int>> adj(n);

    // 构建无向图
    //   0 --- 1 --- 2
    //   |     |
    //   3 --- 4 --- 5
    adj[0] = {1, 3};
    adj[1] = {0, 2, 4};
    adj[2] = {1};
    adj[3] = {0, 4};
    adj[4] = {1, 3, 5};
    adj[5] = {4};

    BFS(adj, 0, n);
    // 输出：0 1 3 2 4 5

    return 0;
}

In [ ]:
main();

**为什么 BFS 要用队列而不是栈？**

- 队列 → FIFO → 先发现的结点先被探索 → **逐层扩展** → BFS
- 栈 → LIFO → 后发现的结点先被探索 → **一条路走到底** → DFS

```
BFS（用队列）像"水波扩散"：

    起点 ●
   第1层  ● ● ●
   第2层  ● ● ● ● ●
   第3层  ● ● ● ● ● ● ●

DFS（用栈）像"走迷宫"：

    起点 ●
           ↓
           ● → ● → ● → ●（走到头）
                         ↓ 回溯
           ● → ●
```

### 6.3 应用三：操作系统中的缓冲区 / 打印队列

操作系统中有很多地方用到队列：

1. **打印队列**：多个用户提交打印任务，按提交顺序排队打印
2. **CPU 调度**：进程按到达时间排队等待 CPU
3. **键盘缓冲区**：按键事件先存入队列，再由程序按顺序处理

```
用户A 提交打印 ──→ ┌──────────────────────┐
用户B 提交打印 ──→ │  打印队列（FIFO）      │ ──→ 打印机
用户C 提交打印 ──→ └──────────────────────┘
```

### 6.4 应用四：求解迷宫最短路径

BFS 天然可以求**无权图的最短路径**，因为它是逐层扩展的，第一次到达终点的路径一定是最短的。

In [ ]:
#include <iostream>
#include <queue>
using namespace std;

In [ ]:
const int ROWS = 5, COLS = 5;
// 0=通路, 1=墙
int maze[ROWS][COLS] = {
    {0, 0, 1, 0, 0},
    {0, 0, 0, 0, 1},
    {1, 0, 1, 0, 0},
    {0, 0, 0, 1, 0},
    {0, 1, 0, 0, 0}
};

In [ ]:
struct Point {
    int x, y, dist;  // 坐标和距离
};

In [ ]:
int dx[] = {-1, 1, 0, 0};
// 上下左右
int dy[] = {0, 0, -1, 1};

In [ ]:
int BFS_Maze(int sx, int sy, int ex, int ey) {
    bool visited[ROWS][COLS] = {false};
    queue<Point> Q;

    visited[sx][sy] = true;
    Q.push({sx, sy, 0});

    while (!Q.empty()) {
        Point cur = Q.front();
        Q.pop();

        // 到达终点
        if (cur.x == ex && cur.y == ey) {
            return cur.dist;
        }

        // 尝试四个方向
        for (int i = 0; i < 4; i++) {
            int nx = cur.x + dx[i];
            int ny = cur.y + dy[i];

            if (nx >= 0 && nx < ROWS && ny >= 0 && ny < COLS
                && maze[nx][ny] == 0 && !visited[nx][ny]) {
                visited[nx][ny] = true;
                Q.push({nx, ny, cur.dist + 1});
            }
        }
    }

    return -1;  // 无法到达
}

In [ ]:
int main() {
    int dist = BFS_Maze(0, 0, 4, 4);
    if (dist != -1) {
        cout << "从(0,0)到(4,4)的最短路径长度：" << dist << endl;
    } else {
        cout << "无法到达！" << endl;
    }
    return 0;
}

In [ ]:
main();

---

## 第七章：C++ STL 中的 queue 和 deque

在实际编程和竞赛中，一般直接使用 STL 而不手写队列。

### 7.1 `queue`（普通队列）

In [ ]:
#include <queue>
using namespace std;

In [ ]:
queue<int> Q;
Q.push(10);
// 入队
Q.push(20);
Q.push(30);
Q.front();
// 获取队头元素 → 10
Q.back();
// 获取队尾元素 → 30
Q.pop();
// 出队（删除队头）
Q.size();
// 队列长度
Q.empty();
// 是否为空

### 7.2 `deque`（双端队列）

In [ ]:
#include <deque>
using namespace std;

In [ ]:
deque<int> dq;
dq.push_back(10);
// 后端入队
dq.push_front(20);
// 前端入队
dq.pop_back();
// 后端出队
dq.pop_front();
// 前端出队
dq.front();
// 前端元素
dq.back();
// 后端元素
dq.size();
// 大小
dq[i];
// 随机访问（deque支持下标访问！）

> **注意**：STL 的 `deque` 底层实现不是简单的循环数组，而是**分段连续的存储结构**（多个固定大小的数组块通过一个中控器管理）。这使得它在两端插入/删除都是 O(1)，中间插入/删除是 O(n)。

### 7.3 `priority_queue`（优先队列 / 堆）

虽然名字里有"队列"，但它**不是 FIFO**，而是按优先级出队（本质是**堆**）。

In [ ]:
#include <queue>
#include <vector>
#include <functional>
using namespace std;

// 默认大顶堆：最大的元素先出队

In [ ]:
priority_queue<int> maxHeap;
maxHeap.push(30);
maxHeap.push(10);
maxHeap.push(20);
maxHeap.top();
// 30（最大值）
maxHeap.pop();
// 移除 30

// 小顶堆：最小的元素先出队
priority_queue<int, vector<int>, greater<int>> minHeap;
minHeap.push(30);
minHeap.push(10);
minHeap.push(20);
minHeap.top();
// 10（最小值）

---

## 第八章：易错点与考试重点总结

### 8.1 高频易错点

**❌ 错误 1：循环队列判满条件搞混**

In [ ]:
// ❌ 错误写法

In [ ]:
if (rear == MAXSIZE) // 这是线性队列的判满，不是循环队列！

// ✅ 正确写法（牺牲一个空间法）
if ((rear + 1) % MAXSIZE == front)

**❌ 错误 2：链队列出队时忘记处理最后一个元素**

In [ ]:
// ❌ 漏掉了 rear 的更新

In [ ]:
bool DeQueue(LinkQueue &Q, int &x) {
    LinkNode *p = Q.front->next;
    x = p->data;
    Q.front->next = p->next;
    delete p;
    return true;
    // 如果删除的是最后一个结点，rear 还指着被 delete 的 p！
    // → 野指针！程序崩溃！
}

// ✅ 正确写法

In [ ]:
bool DeQueue(LinkQueue &Q, int &x) {
    LinkNode *p = Q.front->next;
    x = p->data;
    Q.front->next = p->next;
    if (Q.rear == p) {       // ← 别忘了这行！
        Q.rear = Q.front;
    }
    delete p;
    return true;
}

**❌ 错误 3：不带头结点的链队列入队时忘记特判空队**

In [ ]:
// ❌ 空队时 Q.rear 是 nullptr，Q.rear->next 段错误！

In [ ]:
void EnQueue(LinkQueue &Q, int x) {
    LinkNode *s = new LinkNode;
    s->data = x;
    s->next = nullptr;
    Q.rear->next = s;  // 💥 如果 Q.rear == nullptr
    Q.rear = s;
}

### 8.2 考试核心知识点速查表

| 知识点                     | 内容                                 |
| -------------------------- | ------------------------------------ |
| 队列的特点                 | 先进先出（FIFO）                     |
| 循环队列队空               | `front == rear`                      |
| 循环队列队满（牺牲空间法） | `(rear+1) % MAXSIZE == front`        |
| 循环队列长度               | `(rear - front + MAXSIZE) % MAXSIZE` |
| 链队列队空（带头结点）     | `front == rear`（都指向头结点）      |
| 链队列队空（不带头结点）   | `front == nullptr`                   |
| 双端队列                   | 两端都可入队和出队                   |
| 输入受限双端队列           | 一端入，两端出                       |
| 输出受限双端队列           | 两端入，一端出                       |
| 队列典型应用               | 层序遍历、BFS、CPU调度、缓冲区       |

### 8.3 循环队列的三种判满方案对照

```
方案一：牺牲一个空间
  队满：(rear + 1) % MAXSIZE == front
  最大存储：MAXSIZE - 1 个元素
  优点：不需要额外变量
  ⭐ 最常考

方案二：增设 size
  队满：size == MAXSIZE
  最大存储：MAXSIZE 个元素
  优点：空间利用率最高

方案三：增设 tag
  队满：front == rear && tag == 1
  队空：front == rear && tag == 0
  tag：入队后置1，出队后置0
```

---

## 结语

队列虽然结构简单，但在计算机科学中无处不在：

- **操作系统**：进程调度、I/O缓冲
- **网络**：数据包排队
- **算法**：BFS、层序遍历、拓扑排序
- **日常生活**：排队、叫号系统

掌握队列的核心在于：
1. 理解 **FIFO** 原则
2. 熟练写出 **循环队列** 的各种操作和判断条件
3. 会写 **链队列**（带头结点 + 不带头结点）
4. 理解 **双端队列** 的三种形态
5. 能够将队列应用到 **层序遍历** 和 **BFS** 中

这些知识不仅是考试的重中之重，也是后续学习树、图、操作系统等课程的基础。务必动手把每段代码都跑一遍、改一改、调一调，才能真正理解透彻！